# High-SNR Semantic Direction: Windowed Extraction + Invariance

This notebook re-extracts a candidate **semantic** direction with two fixes and one ablation:

1. **Window fix** — compare on-policy vs off-policy in a matched window:
   - `matched_lines`: mean over response tokens up to the **last edited line**.
   - `k_after_edit`: mean over the **first K tokens after the edited line** (recommended).
2. **Holdout** — compute the direction on `train`, choose layer on `val`, report metrics on `test`.
3. **Invariance check (ablation)** — stratify by paraphrase model and by #edited lines to ensure the vector isn't just a style/length detector.

> **Outputs**: per-layer ROC-AUC/d', histograms, length correlation, best-layer vector JSON.


In [ ]:
# Setup
MODEL = 'Qwen/Qwen3-4B'
OFF_POLICY_PATH = 'data/off_policy.json'  # built with fixed 03_create_off_policy.py
WINDOW = 'k_after_edit'                  # 'matched_lines' or 'k_after_edit'
K_TOKENS = 24
TRAIN_FRAC, VAL_FRAC = 0.7, 0.15
SEED = 123
LAYERS = None  # default: second half
VECTOR_OUT = 'artifacts/vector_windowed.json'


In [ ]:
from pathlib import Path
import json, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
from IPython.display import display


In [ ]:
# Reuse the fixed extractor as a module (if running from repo, you can `pip install -e .` or import directly)
import importlib.util, sys
from types import ModuleType

spec = importlib.util.spec_from_file_location('extract', 'fixed/04_extract_vector.py')
mod = importlib.util.module_from_spec(spec)
sys.modules['extract'] = mod
spec.loader.exec_module(mod)

# Run main-like logic in-process (a tiny wrapper)
class Args:
    def __init__(self):
        self.input = Path(OFF_POLICY_PATH)
        self.output = Path(VECTOR_OUT)
        self.model = MODEL
        self.window = WINDOW
        self.k_tokens = K_TOKENS
        self.layers = LAYERS
        self.train_frac = TRAIN_FRAC
        self.val_frac = VAL_FRAC
        self.seed = SEED

args = Args()


In [ ]:
# Execute extraction & evaluation
mod.main.__wrapped__ if hasattr(mod.main, '__wrapped__') else None
mod.main()


In [ ]:
# Load vector and show summary
with open(VECTOR_OUT) as f:
    vec = json.load(f)
vec['layer'], vec['window'], vec['k_tokens']

In [ ]:
# OPTIONAL: diagnostic correlation with response length (rough length proxy)
# This block re-computes projections on TEST and correlates with token length.
import json
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

with open(OFF_POLICY_PATH) as f:
    data = json.load(f)

model = AutoModelForCausalLM.from_pretrained(args.model, dtype=torch.float16, device_map='auto')
tokenizer = AutoTokenizer.from_pretrained(args.model)

# split like extractor
def split_indices(n, seed, train_frac, val_frac):
    rng = np.random.default_rng(seed)
    idxs = np.arange(n)
    rng.shuffle(idxs)
    n_train = int(n*train_frac)
    n_val = int(n*val_frac)
    train = set(idxs[:n_train]); val = set(idxs[n_train:n_train+n_val]); test = set(idxs[n_train+n_val:])
    return train, val, test

train,val,test = split_indices(len(data), args.seed, args.train_frac, args.val_frac)

# restore vector
import numpy as np
v = np.array(vec['vector'], dtype=np.float32)
v = v / (np.linalg.norm(v) + 1e-8)
best_layer = int(vec['layer'])

proj, lengths, labels = [], [], []
for i, ex in enumerate(data):
    if i not in test: continue
    prompt = ex['prompt_with_template']
    on_rollouts = ex['on_policy']
    for var in ex['off_policy']:
        src = int(var['source_rollout_idx'])
        last_idx = int(var.get('last_edited_line_idx', -1))
        on_full = on_rollouts[src]
        off_full = var.get('text_full', var.get('text_clipped',''))
        (on_start, on_end), (off_start, off_end) = mod.build_windows_for_variant(
            tokenizer, prompt, on_full, off_full, last_idx, args.window, args.k_tokens)
        # compute full hidden states once each
        on_full_text = prompt + on_full
        off_full_text = prompt + off_full
        with torch.no_grad():
            on_inputs = tokenizer(on_full_text, return_tensors='pt').to(model.device)
            off_inputs = tokenizer(off_full_text, return_tensors='pt').to(model.device)
            on_out = model(**on_inputs, output_hidden_states=True, use_cache=False)
            off_out = model(**off_inputs, output_hidden_states=True, use_cache=False)
        # slice
        L = best_layer
        on_h = on_out.hidden_states[L+1][0]
        off_h = off_out.hidden_states[L+1][0]
        on_vec = on_h[on_start:on_end,:].mean(dim=0).cpu().numpy()
        off_vec = off_h[off_start:off_end,:].mean(dim=0).cpu().numpy()
        proj.append(float(v @ on_vec)); lengths.append(on_end-on_start); labels.append(1)
        proj.append(float(v @ off_vec)); lengths.append(off_end-off_start); labels.append(0)

import pandas as pd
import numpy as np
from scipy.stats import pearsonr

df = pd.DataFrame({'proj': proj, 'len': lengths, 'y': labels})
print('Len stats:', df['len'].describe().to_string())
print('Corr(proj,len)=', pearsonr(df['proj'], df['len']))
